<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/04_recommendation_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AstroPlanner AI: Recommendation Engine
## Phase 4 — Ranks targets by visibility, weather, moon, equipment, and light pollution

## Setup

In [1]:
!pip install astropy skyfield requests -q

import sys, os, json, warnings
from collections import defaultdict
from datetime import datetime

import numpy as np
import requests
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git 2>/dev/null || git -C /content/AstroPlanner-AI pull

sys.path.append('/content/AstroPlanner-AI/src')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'
os.makedirs(DATA_DIR, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.4/370.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.1/218.1 kB 16.2 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
from models import WeeklyPlanRequest, UserProfile, TargetType

## Load upstream outputs (notebooks 01, 02, 03)

In [5]:
with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

with open(f'{DATA_DIR}/weekly_sky_conditions.json') as f:
    weekly_sky_conditions = json.load(f)

with open(f'{DATA_DIR}/weekly_visibility.json') as f:
    weekly_visibility = json.load(f)

print(f"User          : {plan_request.user.name}")
print(f"Experience    : {plan_request.user.experience_level.value}")
print(f"Nights loaded : {len(weekly_sky_conditions)} (weather), {len(weekly_visibility)} (visibility)")

User          : Arfa
Experience    : beginner
Nights loaded : 7 (weather), 7 (visibility)


## Light pollution

Bortle scale is collected once in notebook 01 — either a number the user
already knew, or a friendly estimate from the guided self-assessment
(area type + one sanity-check question). No external API/raster lookup
here: no verified free data source was available, and a self-reported
estimate is more honest than a fake or unverified automated one. If the
user skipped it in notebook 01, it's `None` here and the light-pollution
factor is simply dropped from scoring — `score_light_pollution()` already
handles that.

In [6]:
bortle_scale = plan_request.user.bortle_scale
if bortle_scale is not None:
    print(f"Light pollution: using Bortle {bortle_scale} from user profile")
else:
    print("Light pollution: not set on user profile — factor will be dropped from scoring for this run.")

Light pollution: not set on user profile — factor will be dropped from scoring for this run.


## Moon illumination

Per-night value (not per-object — object-level moon *separation* already
comes from notebook 03). Reuses the same cached `de421.bsp` ephemeris
notebook 03 downloaded, so this should load instantly.

In [7]:
from skyfield.api import Loader
from skyfield import almanac

skyfield_loader = Loader(DATA_DIR)
ts = skyfield_loader.timescale()

eph_path = os.path.join(DATA_DIR, 'de421.bsp')
try:
    eph = skyfield_loader('de421.bsp')  # reuses notebook 03's cached file if present
except ValueError as e:
    if "file starts with b'<!DOCTYP'" in str(e) and os.path.exists(eph_path):
        os.remove(eph_path)
        eph = skyfield_loader('de421.bsp')
    else:
        raise


def get_moon_illumination(date_str: str) -> float:
    """
    Fraction of the Moon's disc illuminated at local midnight (UTC-naive
    approximation — fine for a nightly planning score, not for precision
    photometry). 0.0 = new moon, 1.0 = full moon.
    """
    t = ts.utc(int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]), 23, 59, 0)
    return float(almanac.fraction_illuminated(eph, 'moon', t))

## NGC type → TargetType mapping

Used for preference matching. `SC` / `CL+N` / `G+C` don't have a
dedicated bucket in `TargetType` yet — mapped to the closest reasonable
fit. Revisit if `TargetType` grows a generic "star_cluster" category.

In [8]:
NGC_TYPE_TO_TARGET_TYPE = {
    'GX': TargetType.galaxy,
    'OC': TargetType.open_cluster,
    'GC': TargetType.globular_cluster,
    'BN': TargetType.nebula,
    'EN': TargetType.nebula,
    'RN': TargetType.nebula,
    'PN': TargetType.nebula,
    'SNR': TargetType.nebula,
    'SC': TargetType.open_cluster,
    'CL+N': TargetType.open_cluster,
    'G+C': TargetType.galaxy,
}


def classify_target_type(obj: dict) -> "TargetType | None":
    """Maps a visibility-engine object (planet/moon or NGC row) to TargetType."""
    if obj.get('is_solar_system'):
        return TargetType.moon if obj['name'] == 'Moon' else TargetType.planet
    return NGC_TYPE_TO_TARGET_TYPE.get(obj.get('type'))

## Per-factor scoring functions

Each returns a float in [0, 1], higher is better, so they can be blended
directly. Five core factors: visibility, weather, moon, equipment,
light pollution.

In [9]:
def score_visibility(obj: dict) -> float:
    """
    Reuses notebook 03's altitude/brightness logic. Deep-sky objects
    already carried a _score there but it's stripped before saving, so
    it's recomputed here from the same inputs for consistency.
    """
    alt_component = obj['peak_altitude_deg'] / 90
    mag = obj.get('magnitude')
    if mag is None:
        # No magnitude on record (some SNR entries) — assume mid-brightness
        # rather than penalizing or rewarding an unknown.
        mag_component = 0.5
    else:
        mag_component = max(0.0, (15 - mag) / 15)
    return 0.5 * alt_component + 0.5 * mag_component


def score_weather(night_weather: dict) -> float:
    """Directly reuses notebook 02's 0-100 sky quality score."""
    return night_weather['sky_quality']['sky_quality_score'] / 100


def score_moon(obj: dict, moon_illumination_fraction: float) -> float:
    """
    Combines the night's overall moon illumination with this object's
    angular separation from the Moon (from notebook 03). Solar system
    objects (including the Moon itself) aren't penalized by moon
    brightness — a bright moon doesn't hurt planetary/lunar observing.
    Deep-sky objects are penalized more when the moon is both bright AND
    close; a bright moon far from the target is a much smaller problem.
    """
    if obj.get('is_solar_system'):
        return 1.0

    separation = obj.get('moon_separation_deg')
    if separation is None:
        proximity_penalty = 0.5  # unknown separation, assume moderate impact
    else:
        # 0deg separation -> full penalty scaling, 90deg+ -> no penalty
        proximity_penalty = max(0.0, 1 - separation / 90)

    return 1 - (moon_illumination_fraction * proximity_penalty)


def score_equipment(obj: dict, user: UserProfile) -> float:
    """
    How well-matched this target is to the user's aperture and
    experience level. Reuses the same limiting-magnitude formula as
    notebook 03's prefilter (kept local here rather than imported, since
    it's a one-line formula — worth promoting to a shared utils module
    if a third notebook ends up needing it too).

    Beginners score best on objects comfortably brighter than the
    telescope's limit (easy, forgiving targets); advanced users score
    well across the full range the telescope can reach, since they're
    not penalized for chasing faint objects.
    """
    if obj.get('is_solar_system'):
        return 1.0  # aperture-independent within amateur ranges; always well-matched

    mag = obj.get('magnitude')
    if mag is None:
        return 0.5

    limiting_mag = 2.1 + 5 * np.log10(user.telescope.aperture_mm)
    headroom = limiting_mag - mag  # magnitudes of margin before it's too faint

    if user.experience_level.value == 'beginner':
        # Reward comfortable margin, cap at 3 magnitudes of headroom
        return float(np.clip(headroom / 3, 0, 1))
    elif user.experience_level.value == 'intermediate':
        return float(np.clip(0.5 + headroom / 6, 0, 1))
    else:  # advanced
        # Flat, generous score across the whole reachable range —
        # faint targets aren't penalized, only genuinely out-of-range ones.
        return 1.0 if headroom >= 0 else 0.0


def score_light_pollution(obj: dict, bortle: "int | None") -> "float | None":
    """
    Returns None (not 0) when bortle is unknown, so the caller can drop
    this factor from the blend entirely rather than treating "unknown"
    as "worst case". When known: deep-sky objects are hurt by light
    pollution much more than planets/the Moon, which stay bright
    regardless of sky background.
    """
    if bortle is None:
        return None
    if obj.get('is_solar_system'):
        return 1.0
    # Bortle 1 (best) -> 1.0, Bortle 9 (worst) -> ~0.1
    return float(np.clip(1 - (bortle - 1) / 9, 0.1, 1.0))


def compute_preference_multiplier(obj: dict, user: UserProfile) -> float:
    """
    Bounded bonus, not a core weighted factor — see module docstring.
    1.0 = no boost (no preferences set, or no match). 1.15 = matches a
    favorite target category. Capped low deliberately so preferences can
    break ties and nudge rankings without overriding a physically poor
    target (e.g. something barely above the horizon in a bright moon).
    """
    prefs = user.preferences
    if not prefs or not prefs.favorite_targets:
        return 1.0
    target_type = classify_target_type(obj)
    return 1.15 if target_type in prefs.favorite_targets else 1.0

## Composite scoring

Blends the five core factors equally (or equally across whichever are
available, if light pollution is unknown), then applies the preference
multiplier on top.

In [10]:
def score_object(
    obj: dict,
    night_weather: dict,
    moon_illumination_fraction: float,
    user: UserProfile,
    bortle: "int | None",
) -> dict:
    """
    Blends the five core factors (equal weight, or equal weight across
    whichever factors are actually available), then applies the
    preference multiplier on top. Returns the object with scoring
    fields attached so the final JSON is self-explanatory rather than
    just a bare ranked list.
    """
    factor_scores = {
        'visibility': score_visibility(obj),
        'weather': score_weather(night_weather),
        'moon': score_moon(obj, moon_illumination_fraction),
        'equipment': score_equipment(obj, user),
    }

    lp_score = score_light_pollution(obj, bortle)
    if lp_score is not None:
        factor_scores['light_pollution'] = lp_score

    base_score = sum(factor_scores.values()) / len(factor_scores)
    multiplier = compute_preference_multiplier(obj, user)
    final_score = round(base_score * multiplier, 4)

    target_type = classify_target_type(obj)

    return {
        **obj,
        'factor_scores': {k: round(v, 3) for k, v in factor_scores.items()},
        'preference_multiplier': multiplier,
        'target_type': target_type.value if target_type else None,
        'recommendation_score': final_score,
    }


def get_weekly_recommendations(
    plan_request: WeeklyPlanRequest,
    weekly_sky_conditions: list,
    weekly_visibility: list,
    bortle: "int | None",
    top_n: int = 15,
) -> list:
    """
    Joins the two weekly datasets by date, scores every visible object
    for that night, and returns the top_n ranked recommendations per
    night. Nights present in one dataset but not the other are skipped
    with a warning rather than crashing the whole run.
    """
    weather_by_date = {n['date']: n for n in weekly_sky_conditions}
    user = plan_request.user

    recommendations = []
    for night in weekly_visibility:
        date_str = night['date']
        night_weather = weather_by_date.get(date_str)
        if night_weather is None:
            print(f"  Warning: no weather data for {date_str}, skipping.")
            continue

        moon_illum = get_moon_illumination(date_str)

        scored = [
            score_object(obj, night_weather, moon_illum, user, bortle)
            for obj in night['objects']
        ]
        scored.sort(key=lambda o: o['recommendation_score'], reverse=True)

        recommendations.append({
            'date': date_str,
            'day_offset': night['day_offset'],
            'moon_illumination_pct': round(moon_illum * 100, 1),
            'sky_quality_verdict': night_weather['sky_quality']['verdict'],
            'bortle_scale_used': bortle,
            'recommended_objects': scored[:top_n],
        })

    return recommendations

In [11]:
weekly_recommendations = get_weekly_recommendations(
    plan_request, weekly_sky_conditions, weekly_visibility, bortle_scale
)

with open(f'{DATA_DIR}/weekly_recommendations.json', 'w') as f:
    json.dump(weekly_recommendations, f, indent=2, default=str)

print(f"\nSaved to {DATA_DIR}/weekly_recommendations.json")


Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_recommendations.json


In [ ]:
best = max(
    weekly_recommendations,
    key=lambda n: n['recommended_objects'][0]['recommendation_score'] if n['recommended_objects'] else 0,
)

print(f"\nBest night: {best['date']}  "
      f"(moon {best['moon_illumination_pct']}% illuminated, sky '{best['sky_quality_verdict']}')\n")

print("=== Top Recommendations ===")
for obj in best['recommended_objects'][:10]:
    pref_flag = "  ★ preferred" if obj['preference_multiplier'] > 1.0 else ""
    factors_str = ', '.join(f'{k}={v}' for k, v in obj['factor_scores'].items())
    print(
        f"  {obj['name']:10} "
        f"score={obj['recommendation_score']:.3f}  "
        f"type={obj.get('target_type') or 'n/a':18} "
        f"[{factors_str}]"
        f"{pref_flag}"
    )